# Пилотная ретроспективная проверка фитофтороза картофеля

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkonov2/AgroPhenology/blob/main/notebooks/late_blight_pilot_validation.ipynb)

Здесь параллельно, но **без смешивания правил**, считаются Hutton Criteria и упрощённая 10-дневная модель Полякова. Вход — два ограниченных сообщения 2026 года: обнаружение в районе деревни Бунятино 24.07 и единичный отрицательный осмотр 20 га 24.08.

Это пилотная проверка отдельных случаев, не оценка точности. Нельзя считать TP/TN/FP/FN, accuracy, sensitivity или specificity. Погодный сигнал не доказывает присутствие патогена, а отсутствие сигнала не доказывает отсутствие болезни.

In [ ]:
from pathlib import Path
import importlib.util, os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
REPOSITORY_URL = 'https://github.com/vkonov2/AgroPhenology.git'
if IN_COLAB:
    repo_root = Path('/content/AgroPhenology')
    if not (repo_root / 'pyproject.toml').exists():
        try:
            subprocess.check_call(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(repo_root)])
        except subprocess.CalledProcessError as exc:
            raise RuntimeError('Не удалось клонировать репозиторий. Для закрытого репозитория загрузите архив вручную; токены в notebook не добавляйте.') from exc
else:
    repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
    if repo_root is None:
        raise RuntimeError('Запустите notebook из checkout репозитория.')
if importlib.util.find_spec('agro_phenology') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_root)])
source_path = str(repo_root / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
os.chdir(repo_root)
print(f'Режим: {"Google Colab" if IN_COLAB else "локальный Jupyter"}; репозиторий: {repo_root}')

## Зафиксированные допущения

Для положительной записи дана координата деревни Бунятино `56.4008889, 37.2483611`; это координата населённого пункта, а не подтверждённая координата поля. Для отрицательной записи используется явно маркированный прокси центра Сергиева Посада.

Для Hutton заранее показываются окна 7, 14 и 21 день; «лучшее» окно не выбирается. Погода дня осмотра исключена из правил ассоциации, потому что время осмотра неизвестно. Автор данных подтвердил фазу бутонизации и региональный интервал `15–30.06.2026`; точная полевая дата неизвестна. Для Полякова рассчитываются все 16 дат интервала без выбора и голосования между сценариями.

In [ ]:
from datetime import datetime, timezone
import json, shutil
import pandas as pd
from IPython.display import display

from agro_phenology.open_meteo import OpenMeteoClient
from agro_phenology.late_blight_pilot import run_late_blight_pilot
from agro_phenology.plotting import plot_late_blight_pilot_timeline

observations_path = repo_root / 'data/examples/late_blight_observations_pilot.csv'
observations = pd.read_csv(observations_path)
display(observations)
print('ВНИМАНИЕ: это две пилотные метки из сообщения, не выгрузка мониторинга.')

client = OpenMeteoClient(repo_root / 'data/cache/open_meteo')
pilot = run_late_blight_pilot(observations, client)
print(f"Условных точек анализа: {len(pilot['analysis_points'])}")
display(pilot['analysis_points'][['observation_id', 'analysis_point_id', 'analysis_latitude', 'analysis_longitude', 'coordinate_rule']])

In [ ]:
weather_columns = [
    'analysis_point_id', 'requested_end_date',
    'era5_land_returned_latitude', 'era5_land_returned_longitude',
    'era5_returned_latitude', 'era5_returned_longitude',
    'temperature_valid_hour_count', 'humidity_valid_hour_count',
    'precipitation_valid_hour_count', 'last_temperature_local_time',
    'last_precipitation_local_time', 'temperature_source_model',
    'precipitation_source_model', 'source_alignment_status',
    'era5_land_cache_hit', 'era5_precipitation_cache_hit',
]
display(pilot['weather_metadata'][weather_columns])
print('Профиль: T/RH — ERA5-Land; осадки — отдельный ERA5. Временные оси должны совпадать точно.')
negative_weather = pilot['weather_metadata'][pilot['weather_metadata']['observation_id'].eq('LB-PILOT-002')]
if not negative_weather.empty:
    print('Последнее доступное локальное время T/RH для записи 24.08:', negative_weather.iloc[0]['last_temperature_local_time'])
    print('Последнее доступное локальное время осадков:', negative_weather.iloc[0]['last_precipitation_local_time'])

In [ ]:
hutton = pilot['case_results'][pilot['case_results']['model'].eq('hutton_criteria')].copy()
hutton['signal_true'] = hutton['hutton_signal_present'].eq(True)
hutton['signal_indeterminate'] = hutton['hutton_signal_present'].isna()
window_summary = hutton.groupby(['observation_id', 'lookback_days'], as_index=False).agg(
    point_count=('analysis_point_id', 'nunique'),
    points_with_signal=('signal_true', 'sum'),
    indeterminate_points=('signal_indeterminate', 'sum'),
    nearest_lag_min_days=('days_from_nearest_prior_hutton_period', 'min'),
    nearest_lag_max_days=('days_from_nearest_prior_hutton_period', 'max'),
)
window_summary['signal_fraction_all_points'] = window_summary['points_with_signal'] / window_summary['point_count']
display(window_summary)
display(hutton[[
    'observation_id', 'analysis_point_id', 'lookback_days', 'hutton_signal_present',
    'hutton_pass_pair_count', 'hutton_indeterminate_pair_count',
    'days_from_nearest_prior_hutton_period', 'pilot_association',
]])

In [ ]:
polyakov = pilot['case_results'][pilot['case_results']['model'].eq('polyakov_late_blight_10d_v1')].copy()
polyakov_interval = polyakov[polyakov['result_scope'].eq('primary_activation_interval_summary')].copy()
display(polyakov_interval[[
    'observation_id', 'analysis_point_id', 'phenophase_name', 'phenophase_status',
    'activation_date_start', 'activation_date_end', 'field_specific_activation_date_known',
    'polyakov_evaluability_status', 'activation_scenario_count',
    'evaluable_activation_scenario_count', 'activation_interval_status',
    'activation_interval_result', 'nearest_prior_manifestation_start_min',
    'nearest_prior_manifestation_start_max', 'nearest_prior_manifestation_end_min',
    'nearest_prior_manifestation_end_max',
    'days_from_nearest_prior_manifestation_end_min',
    'days_from_nearest_prior_manifestation_end_max',
]])
polyakov_scenarios = polyakov[polyakov['result_scope'].eq('primary_activation_interval_scenario')]
display(polyakov_scenarios.groupby(['observation_id', 'pilot_association'], as_index=False).size())
for row in polyakov_interval.itertuples(index=False):
    print(f'{row.observation_id}: {row.activation_interval_result}')

In [ ]:
results_dir = repo_root / 'results/late_blight_pilot'
results_dir.mkdir(parents=True, exist_ok=True)
pilot['case_results'].to_csv(results_dir / 'case_results.csv', index=False)
pilot['daily_features'].to_csv(results_dir / 'daily_features.csv', index=False)
pilot['model_events'].to_csv(results_dir / 'model_events.csv', index=False)
pilot['weather_metadata'].to_csv(results_dir / 'weather_metadata.csv', index=False)
methodology = {
    'execution_timestamp': datetime.now(timezone.utc).isoformat(),
    'weather_provider': 'Open-Meteo Historical Weather API',
    'weather_sources': {
        'temperature_2m': {'dataset': 'ERA5-Land', 'model': 'era5_land'},
        'relative_humidity_2m': {'dataset': 'ERA5-Land', 'model': 'era5_land'},
        'precipitation': {'dataset': 'ERA5', 'model': 'era5'},
    },
    'weather_source_profile': pilot['configuration']['weather_source_profile'],
    'api_requests': {
        'temperature_humidity': {'models': 'era5_land', 'hourly': ['temperature_2m', 'relative_humidity_2m']},
        'precipitation': {'models': 'era5', 'hourly': ['precipitation'], 'precipitation_unit': 'mm'},
        'shared': {'timezone': 'auto', 'cell_selection': 'land'},
    },
    'source_merge': {
        'key': 'exact local hourly timestamp', 'missing_policy': 'preserve_na',
        'timezone_policy': 'exact_match', 'interpolation': 'forbidden',
    },
    'configuration': pilot['configuration'],
    'hutton_lookback_days': [7, 14, 21],
    'observation_day_excluded': True,
    'source_specifications': ['hutton_criteria_pipeline_ru.pdf', 'fitoftoroz_kartofelya_pipeline_polyakov.pdf'],
    'coordinate_assumption': 'Bunyatino village coordinate and Sergiev Posad center proxy; neither confirmed field coordinate',
    'phenophase_interval': {
        'stage': 'budding', 'status': 'AUTHOR_CONFIRMED_REGIONAL_INTERVAL',
        'start_date': '2026-06-15', 'end_date': '2026-06-30',
        'field_specific_activation_date_known': False,
        'policy': 'evaluate every date in interval without selection or majority voting',
    },
    'specification_resolutions': {
        'hutton_and_indeterminate': 'fail_dominates_indeterminate',
        'polyakov_daily_coverage': 'all expected local-day hours required',
    },
    'polyakov_primary_status': 'conditionally evaluable over author-confirmed regional activation interval',
    'polyakov_interval_results': polyakov_interval[[
        'observation_id', 'polyakov_evaluability_status',
        'activation_interval_status', 'activation_interval_result',
    ]].to_dict(orient='records'),
    'interpretation': 'pilot case association, not model accuracy',
}
(results_dir / 'methodology.json').write_text(json.dumps(methodology, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
timeline_path = results_dir / 'timeline.png'
fig = plot_late_blight_pilot_timeline(pilot['daily_features'], pilot['period_features'], observations, timeline_path)
display(fig)
print('Создан один итоговый график:', timeline_path)

In [ ]:
archive_path = Path(shutil.make_archive(str(repo_root / 'late_blight_pilot_results'), 'zip', results_dir))
print('Архив результатов:', archive_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))

## Как интерпретировать

Совпадение Hutton с одним из заранее заданных временных окон — только временная ассоциация. `SIGNAL_WITHOUT_DETECTION` не называется ложноположительным результатом: могли отсутствовать инокулюм или симптомы, могли применяться обработки, а единичный осмотр мог не обнаружить болезнь.

Для настоящей валидации нужны точные координаты обследованных полей, точная полевая дата бутонизации вместо регионального интервала, время и протокол осмотра, первое обнаружение, сорт, обработки и полный сезон регулярных независимых осмотров.